# Dimension Profiling – STATS19 OLAP Schema

Profiling toàn bộ **12 Dimension** theo thiết kế Snowflake Schema.

Với mỗi Dimension:
1. Xác định **grain** (composite key) theo schema.
2. Tạo tổ hợp **DISTINCT** theo grain đó.
3. Đếm **Distinct Rows** và **Unknown Rows**.
4. Tính **Expected Rows** (= Distinct Rows từ data).

> Chỉ đọc file đã clean trong `Data/Processed/` — KHÔNG đọc file Raw.

---

## 0. Setup & Load Data

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path(r'D:\OLAP\Data\Processed')
OUT_DIR  = Path(r'D:\OLAP\Data\Profiling')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Loading processed files...')
collisions = pd.read_csv(
    BASE_DIR / 'collisions_clean.csv',
    dtype={'collision_id': str, 'local_authority_district_code': str}
)
vehicles = pd.read_csv(
    BASE_DIR / 'vehicles_clean.csv',
    dtype={'collision_id': str, 'vehicle_reference': str}
)
casualties = pd.read_csv(
    BASE_DIR / 'casualties_clean.csv',
    dtype={'collision_id': str, 'vehicle_reference': str, 'casualty_reference': str}
)
print(f'  collisions : {len(collisions):,} rows')
print(f'  vehicles   : {len(vehicles):,} rows')
print(f'  casualties : {len(casualties):,} rows')
print('Done.')

Loading processed files...
  collisions : 513,801 rows
  vehicles   : 937,265 rows
  casualties : 652,821 rows
Done.


---
## 1. Helper Functions

In [18]:
UNKNOWN_KW = [
    'unknown', 'not known', 'not reported',
    'data missing or out of range', 'not applicable', 'undefined'
]

def count_unknown_rows(df, grain_cols):
    """So dong DISTINCT co it nhat 1 cot chua Unknown hoac NULL."""
    distinct = df[grain_cols].drop_duplicates()
    mask = pd.Series(False, index=distinct.index)
    for col in grain_cols:
        if distinct[col].dtype == object:
            lower = distinct[col].fillna('').str.strip().str.lower()
            for kw in UNKNOWN_KW:
                mask |= lower.str.contains(kw, na=False)
        mask |= distinct[col].isna()
    return int(mask.sum())


results = []  # tich luy ket qua cho bao cao tong hop

def profile(dim_name, df, grain_cols, notes=''):
    """Tinh Distinct / Unknown / Expected rows, in ket qua va luu vao results."""
    distinct_rows = len(df[grain_cols].drop_duplicates())
    unknown_rows  = count_unknown_rows(df, grain_cols)
    results.append({
        'Dimension'    : dim_name,
        'Grain Columns': ', '.join(grain_cols),
        'Distinct Rows': distinct_rows,
        'Unknown Rows' : unknown_rows,
        'Expected Rows': distinct_rows,
        'Notes'        : notes,
    })
    print(f'{dim_name}')
    print(f'  Grain        : {grain_cols}')
    print(f'  Distinct Rows: {distinct_rows:,}')
    print(f'  Unknown Rows : {unknown_rows:,}')
    print(f'  Expected Rows: {distinct_rows:,}')
    if notes:
        print(f'  Notes        : {notes}')
    print()

print('Helpers defined.')

Helpers defined.


---
## 2. Chiều Thời gian

### Schema
| Bảng | Grain |
|---|---|
| DimDate | collision_date (mỗi ngày duy nhất = 1 row) |
| DimTime | collision_time HH:MM (mỗi phút duy nhất = 1 row) |
| DimTimeBand | Static – 6 bands + 1 Unknown |

### 2.1 DimDate
**Grain**: `collision_date` – mỗi ngày calendar duy nhất = 1 row.

In [19]:
# Parse date, xem distinct
collisions['_date'] = pd.to_datetime(collisions['collision_date'], errors='coerce')
date_distinct = collisions[['collision_date']].drop_duplicates().reset_index(drop=True)

null_dates = collisions['collision_date'].isna().sum()
parse_fail = collisions['_date'].isna().sum() - null_dates

print(f'DimDate – Distinct dates in data: {len(date_distinct):,}')
print(f'NULL collision_date      : {null_dates:,}')
print(f'Parse failures           : {parse_fail:,}')
print(f'Date range: {collisions["_date"].min().date()} to {collisions["_date"].max().date()}')

profile(
    'DimDate', collisions, ['collision_date'],
    notes='1 row per unique date; key = YYYYMMDD INT; Key 0 for Unknown'
)

DimDate – Distinct dates in data: 1,826
NULL collision_date      : 0
Parse failures           : 0
Date range: 2021-01-01 to 2025-12-31
DimDate
  Grain        : ['collision_date']
  Distinct Rows: 1,826
  Unknown Rows : 0
  Expected Rows: 1,826
  Notes        : 1 row per unique date; key = YYYYMMDD INT; Key 0 for Unknown



### 2.2 DimTime
**Grain**: `collision_time` (HH:MM) – mỗi phút duy nhất = 1 row.

In [20]:
time_distinct = collisions[['collision_time']].drop_duplicates().reset_index(drop=True)
null_times = collisions['collision_time'].isna().sum()

collisions['_hour'] = pd.to_datetime(
    collisions['collision_time'], format='%H:%M', errors='coerce'
).dt.hour
parse_fail_t = collisions['_hour'].isna().sum() - null_times

print(f'DimTime – Distinct times (HH:MM) in data: {len(time_distinct):,}')
print(f'NULL collision_time : {null_times:,}')
print(f'Parse failures      : {parse_fail_t:,}')
print(f'Max possible (1440 min/day): 1,440')

profile(
    'DimTime', collisions, ['collision_time'],
    notes='1 row per unique HH:MM; max 1440 rows; FK -> DimTimeBand'
)

DimTime – Distinct times (HH:MM) in data: 1,440
NULL collision_time : 0
Parse failures      : 513,801
Max possible (1440 min/day): 1,440
DimTime
  Grain        : ['collision_time']
  Distinct Rows: 1,440
  Unknown Rows : 0
  Expected Rows: 1,440
  Notes        : 1 row per unique HH:MM; max 1440 rows; FK -> DimTimeBand



### 2.3 DimTimeBand *(Static)*
**Grain**: Static – 6 time bands + 1 Unknown placeholder.

In [21]:
timeband_static = pd.DataFrame({
    'TimeBandKey' : [0, 1, 2, 3, 4, 5, 6],
    'TimeBandName': [
        'Unknown',
        'Night (00-05)',
        'Early Morning (06-08)',
        'Morning Peak (09-11)',
        'Afternoon (12-16)',
        'Evening Peak (17-19)',
        'Evening (20-23)'
    ],
    'TimeBandSort': [0, 1, 2, 3, 4, 5, 6]
})
display(timeband_static)

# Validate: tat ca gio trong data co duoc map day du khong
hour_coverage = collisions['_hour'].dropna().unique()
print(f'Hours present in data (0-23): {sorted(hour_coverage)}')

# Them thu cong vao results (static)
results.append({
    'Dimension'    : 'DimTimeBand',
    'Grain Columns': 'TimeBandName (static)',
    'Distinct Rows': 6,
    'Unknown Rows' : 1,
    'Expected Rows': 7,
    'Notes'        : 'Static – 6 bands + 1 Unknown (Key=0) placeholder',
})
print('DimTimeBand')
print('  Distinct Rows: 6')
print('  Unknown Rows : 1')
print('  Expected Rows: 7 (6 bands + 1 Unknown)')

,TimeBandKey,TimeBandName,TimeBandSort
0,0,Unknown,0
1,1,Night (00-05),1
2,2,Early Morning (06-08),2
3,3,Morning Peak (09-11),3
4,4,Afternoon (12-16),4
5,5,Evening Peak (17-19),5
6,6,Evening (20-23),6


Hours present in data (0-23): []
DimTimeBand
  Distinct Rows: 6
  Unknown Rows : 1
  Expected Rows: 7 (6 bands + 1 Unknown)


---
## 3. Chiều Không gian

### Schema
| Bảng | Grain |
|---|---|
| DimLocalAuthority | local_authority_district_code |
| DimLocation | longitude, latitude, local_authority_district_code |

### 3.1 DimLocalAuthority *(Sub-dimension)*
**Grain**: `local_authority_district_code`

In [22]:
la_distinct = collisions[['local_authority_district_code']].drop_duplicates().reset_index(drop=True)
print(f'DimLocalAuthority – Distinct codes: {len(la_distinct):,}')
print(f'Sample (first 10):')
display(la_distinct.head(10))

profile(
    'DimLocalAuthority', collisions, ['local_authority_district_code'],
    notes='Code only – LocalAuthorityName & RegionName need ONS reference table (NOT in clean data)'
)

DimLocalAuthority – Distinct codes: 370
Sample (first 10):


,local_authority_district_code
0,E06000002
1,E06000004
2,E06000005
3,E06000006
4,E06000007
5,E06000008
6,E06000009
7,E06000010
8,E06000011
9,E06000012


DimLocalAuthority
  Grain        : ['local_authority_district_code']
  Distinct Rows: 370
  Unknown Rows : 0
  Expected Rows: 370
  Notes        : Code only – LocalAuthorityName & RegionName need ONS reference table (NOT in clean data)



### 3.2 DimLocation
**Grain**: `longitude × latitude × local_authority_district_code`

In [23]:
grain_loc = ['longitude', 'latitude', 'local_authority_district_code']
loc_distinct = collisions[grain_loc].drop_duplicates().reset_index(drop=True)

null_lon = collisions['longitude'].isna().sum()
null_lat = collisions['latitude'].isna().sum()
print(f'DimLocation – Distinct combinations: {len(loc_distinct):,}')
print(f'NULL longitude  : {null_lon:,}')
print(f'NULL latitude   : {null_lat:,}')
print(f'Sample (first 5):')
display(loc_distinct.head(5))

profile(
    'DimLocation', collisions, grain_loc,
    notes='NULL coords allowed (DECIMAL NULL) – need Unknown placeholder row; FK -> DimLocalAuthority'
)

DimLocation – Distinct combinations: 512,359
NULL longitude  : 53
NULL latitude   : 53
Sample (first 5):


,longitude,latitude,local_authority_district_code
0,-1.23119,54.56929,E06000002
1,-1.31046,54.57776,E06000004
2,-1.57237,54.53082,E06000005
3,-2.72862,53.34161,E06000006
4,-2.73589,53.34121,E06000006


DimLocation
  Grain        : ['longitude', 'latitude', 'local_authority_district_code']
  Distinct Rows: 512,359
  Unknown Rows : 33
  Expected Rows: 512,359
  Notes        : NULL coords allowed (DECIMAL NULL) – need Unknown placeholder row; FK -> DimLocalAuthority



---
## 4. Chiều Đặc điểm Đường & Môi trường

### Schema
| Bảng | Grain |
|---|---|
| DimRoad | first_road_class, road_type, speed_limit_mph, area_type |
| DimEnvironment | light_conditions, weather_conditions, road_surface_conditions |
| DimJunction | junction_detail, junction_control, pedestrian_crossing |

### 4.1 DimRoad
**Grain**: `first_road_class × road_type × speed_limit_mph × area_type`

In [24]:
grain_road = ['first_road_class', 'road_type', 'speed_limit_mph', 'area_type']
road_distinct = collisions[grain_road].drop_duplicates().reset_index(drop=True)

for col in grain_road:
    vals = sorted(collisions[col].dropna().unique())
    print(f'  [{col}] ({len(vals)} unique values): {vals}')
print()
print(f'NULL speed_limit_mph: {road_distinct["speed_limit_mph"].isna().sum()}')
display(road_distinct.head(10))

profile(
    'DimRoad', collisions, grain_road,
    notes='speed_limit_mph is numeric – NULL row counted as Unknown'
)

  [first_road_class] (7 unique values): ['A', 'A(M)', 'B', 'C', 'Motorway', 'Unclassified', 'Unknown']
  [road_type] (6 unique values): ['Dual carriageway', 'One way street', 'Roundabout', 'Single carriageway', 'Slip road', 'Unknown']
  [speed_limit_mph] (6 unique values): [np.float64(20.0), np.float64(30.0), np.float64(40.0), np.float64(50.0), np.float64(60.0), np.float64(70.0)]
  [area_type] (4 unique values): ['Rural', 'Unallocated', 'Unknown', 'Urban']

NULL speed_limit_mph: 2


,first_road_class,road_type,speed_limit_mph,area_type
0,Unclassified,Single carriageway,30.0,Urban
1,A,Single carriageway,30.0,Urban
2,B,Single carriageway,30.0,Urban
3,A,Dual carriageway,40.0,Urban
4,A,Dual carriageway,30.0,Urban
5,A,Slip road,70.0,Urban
6,Unclassified,Single carriageway,20.0,Urban
7,Unclassified,Roundabout,30.0,Rural
8,A,Roundabout,30.0,Urban
9,A,Dual carriageway,40.0,Rural


DimRoad
  Grain        : ['first_road_class', 'road_type', 'speed_limit_mph', 'area_type']
  Distinct Rows: 389
  Unknown Rows : 2
  Expected Rows: 389
  Notes        : speed_limit_mph is numeric – NULL row counted as Unknown



### 4.2 DimEnvironment *(Junk Dimension)*
**Grain**: `light_conditions × weather_conditions × road_surface_conditions`

In [25]:
grain_env = ['light_conditions', 'weather_conditions', 'road_surface_conditions']
env_distinct = collisions[grain_env].drop_duplicates().reset_index(drop=True)

for col in grain_env:
    vals = sorted(collisions[col].dropna().unique())
    print(f'  [{col}] ({len(vals)} unique values): {vals}')
print()
display(env_distinct.head(10))

profile(
    'DimEnvironment', collisions, grain_env,
    notes='Junk dimension – cross product of 3 environment cols'
)

  [light_conditions] (5 unique values): ['Darkness - lights lit', 'Darkness - lights unlit', 'Darkness - no lighting', 'Daylight', 'Unknown']
  [weather_conditions] (9 unique values): ['Fine + high winds', 'Fine no high winds', 'Fog or mist', 'Other', 'Raining + high winds', 'Raining no high winds', 'Snowing + high winds', 'Snowing no high winds', 'Unknown']
  [road_surface_conditions] (6 unique values): ['Dry', 'Flood over 3cm. deep', 'Frost or ice', 'Snow', 'Unknown', 'Wet or damp']



,light_conditions,weather_conditions,road_surface_conditions
0,Darkness - lights lit,Fine no high winds,Dry
1,Daylight,Fine no high winds,Dry
2,Darkness - lights lit,Raining no high winds,Wet or damp
3,Daylight,Raining no high winds,Dry
4,Unknown,Raining no high winds,Wet or damp
5,Daylight,Fine no high winds,Wet or damp
6,Daylight,Raining no high winds,Wet or damp
7,Daylight,Unknown,Wet or damp
8,Darkness - lights lit,Other,Wet or damp
9,Daylight,Other,Dry


DimEnvironment
  Grain        : ['light_conditions', 'weather_conditions', 'road_surface_conditions']
  Distinct Rows: 228
  Unknown Rows : 0
  Expected Rows: 228
  Notes        : Junk dimension – cross product of 3 environment cols



### 4.3 DimJunction *(Junk Dimension)*
**Grain**: `junction_detail × junction_control × pedestrian_crossing`

In [26]:
grain_junc = ['junction_detail', 'junction_control', 'pedestrian_crossing']
junc_distinct = collisions[grain_junc].drop_duplicates().reset_index(drop=True)

for col in grain_junc:
    vals = sorted(collisions[col].dropna().unique())
    print(f'  [{col}] ({len(vals)} unique values): {vals}')
print()
display(junc_distinct.head(10))

profile(
    'DimJunction', collisions, grain_junc,
    notes='Junk dimension – 3 junction / crossing cols'
)

  [junction_detail] (7 unique values): ['Crossroads', 'Junction with more than four arms (not roundabout)', 'Not at junction or within 20 metres', 'Other junction', 'T or staggered junction', 'Unknown', 'Using private drive or entrance']
  [junction_control] (5 unique values): ['Authorised person', 'Auto traffic signal', 'Give way or uncontrolled', 'Stop sign', 'Unknown']
  [pedestrian_crossing] (9 unique values): ['Central refuge - no other controls', 'Footbridge or subway', 'Human crossing control by other authorised person', 'Human crossing control by school crossing patrol', 'No physical crossing facility within 50m', 'Pedestrian light crossing (pelican or puffin or toucan or similar)', 'Pedestrian phase at traffic signal', 'Unknown', 'Zebra crossing']



,junction_detail,junction_control,pedestrian_crossing
0,Crossroads,Give way or uncontrolled,No physical crossing facility within 50m
1,Not at junction or within 20 metres,Unknown,No physical crossing facility within 50m
2,T or staggered junction,Give way or uncontrolled,No physical crossing facility within 50m
3,Crossroads,Give way or uncontrolled,Zebra crossing
4,T or staggered junction,Stop sign,No physical crossing facility within 50m
5,Crossroads,Give way or uncontrolled,Pedestrian light crossing (pelican or puffin or toucan or similar)
6,Not at junction or within 20 metres,Unknown,Pedestrian phase at traffic signal
7,Other junction,Give way or uncontrolled,Human crossing control by other authorised person
8,Other junction,Give way or uncontrolled,No physical crossing facility within 50m
9,Unknown,Give way or uncontrolled,No physical crossing facility within 50m


DimJunction
  Grain        : ['junction_detail', 'junction_control', 'pedestrian_crossing']
  Distinct Rows: 295
  Unknown Rows : 0
  Expected Rows: 295
  Notes        : Junk dimension – 3 junction / crossing cols



---
## 5. Chiều Đối tượng (Vehicle & Casualty)

### Schema
| Bảng | Grain |
|---|---|
| DimPedestrianContext | pedestrian_location, pedestrian_movement |
| DimCasualtyProfile | casualty_age_band, casualty_sex, casualty_class, casualty_type + FK to DimPedestrianContext |
| DimVehicleEvent | vehicle_type, driver_age_band, driver_sex, vehicle_manoeuvre, skidding_and_overturning, vehicle_leaving_carriageway |

### 5.1 DimPedestrianContext *(Sub-dimension)*
**Grain**: `pedestrian_location × pedestrian_movement`

In [27]:
grain_ped = ['pedestrian_location', 'pedestrian_movement']
ped_distinct = casualties[grain_ped].drop_duplicates().reset_index(drop=True)

for col in grain_ped:
    vals = sorted(casualties[col].dropna().unique())
    print(f'  [{col}] ({len(vals)} unique values): {vals}')
print()
display(ped_distinct)

profile(
    'DimPedestrianContext', casualties, grain_ped,
    notes='Sub-dimension of DimCasualtyProfile (Snowflake branch)'
)

  [pedestrian_location] (11 unique values): ['Crossing elsewhere within 50m. of pedestrian crossing', 'Crossing in zig-zag approach lines', 'Crossing in zig-zag exit lines', 'Crossing on pedestrian crossing facility', 'In carriageway, crossing elsewhere', 'In carriageway, not crossing', 'In centre of carriageway - not on refuge, island or central reservation', 'Not a Pedestrian', 'On footway or verge', 'On refuge, central island or central reservation', 'Unknown']
  [pedestrian_movement] (10 unique values): ["Crossing from driver's nearside", "Crossing from driver's offside", 'Crossing from nearside - masked by parked or stationary vehicle', 'Crossing from offside - masked by parked or stationary vehicle', 'In carriageway, stationary - not crossing (standing or playing)', 'In carriageway, stationary - not crossing (standing or playing) - masked by parked or stationary vehicle', 'Not a Pedestrian', 'Unknown', 'Walking along in carriageway, back to traffic', 'Walking along in carriageway

,pedestrian_location,pedestrian_movement
0,"In centre of carriageway - not on refuge, island or central reservation",Crossing from driver's nearside
1,"In carriageway, crossing elsewhere",Crossing from driver's nearside
2,"In carriageway, crossing elsewhere",Unknown
3,On footway or verge,Unknown
4,Crossing elsewhere within 50m. of pedestrian crossing,Crossing from driver's nearside
...,...,...
84,Crossing on pedestrian crossing facility,Not a Pedestrian
85,Crossing in zig-zag approach lines,"In carriageway, stationary - not crossing (standing or playing)"
86,Crossing on pedestrian crossing facility,"Walking along in carriageway, back to traffic"
87,Not a Pedestrian,Crossing from driver's nearside


DimPedestrianContext
  Grain        : ['pedestrian_location', 'pedestrian_movement']
  Distinct Rows: 89
  Unknown Rows : 0
  Expected Rows: 89
  Notes        : Sub-dimension of DimCasualtyProfile (Snowflake branch)



### 5.2 DimCasualtyProfile
**Grain**: `casualty_age_band × casualty_sex × casualty_class × casualty_type × pedestrian_location × pedestrian_movement`

> Grain bao gồm cả cột pedestrian vì DimCasualtyProfile có FK → DimPedestrianContext.

In [28]:
grain_cas = [
    'casualty_age_band', 'casualty_sex', 'casualty_class',
    'casualty_type', 'pedestrian_location', 'pedestrian_movement'
]
cas_distinct = casualties[grain_cas].drop_duplicates().reset_index(drop=True)

for col in grain_cas:
    vals = sorted(casualties[col].dropna().unique())
    print(f'  [{col}] ({len(vals)} unique values): {vals}')
print()
display(cas_distinct.head(10))

profile(
    'DimCasualtyProfile', casualties, grain_cas,
    notes='Grain includes pedestrian cols (FK to DimPedestrianContext)'
)

  [casualty_age_band] (12 unique values): ['0 - 5', '11 - 15', '16 - 20', '21 - 25', '26 - 35', '36 - 45', '46 - 55', '56 - 65', '6 - 10', '66 - 75', 'Over 75', 'Unknown']
  [casualty_sex] (3 unique values): ['Female', 'Male', 'Unknown']
  [casualty_class] (3 unique values): ['Driver or rider', 'Passenger', 'Pedestrian']
  [casualty_type] (20 unique values): ['Agricultural vehicle occupant', 'Bus or coach occupant (17 or more pass seats)', 'Car occupant', 'Cyclist', 'Electric motorcycle rider or passenger', 'Goods vehicle (7.5 tonnes mgw and over) occupant', 'Goods vehicle (over 3.5t. and under 7.5t.) occupant', 'Horse rider', 'Minibus (8 - 16 passenger seats) occupant', 'Mobility scooter rider', 'Motorcycle 125cc and under rider or passenger', 'Motorcycle 50cc and under rider or passenger', 'Motorcycle over 125cc and up to 500cc rider or passenger', 'Motorcycle over 500cc rider or passenger', 'Other vehicle occupant', 'Pedestrian', 'Taxi/Private hire car occupant', 'Tram occupant', 'U

,casualty_age_band,casualty_sex,casualty_class,casualty_type,pedestrian_location,pedestrian_movement
0,46 - 55,Male,Pedestrian,Pedestrian,"In centre of carriageway - not on refuge, island or central reservation",Crossing from driver's nearside
1,26 - 35,Male,Pedestrian,Pedestrian,"In carriageway, crossing elsewhere",Crossing from driver's nearside
2,6 - 10,Male,Pedestrian,Pedestrian,"In carriageway, crossing elsewhere",Unknown
3,36 - 45,Male,Pedestrian,Pedestrian,On footway or verge,Unknown
4,11 - 15,Male,Pedestrian,Pedestrian,Crossing elsewhere within 50m. of pedestrian crossing,Crossing from driver's nearside
5,6 - 10,Male,Pedestrian,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's offside
6,11 - 15,Male,Pedestrian,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's nearside
7,6 - 10,Female,Pedestrian,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's nearside
8,11 - 15,Female,Pedestrian,Pedestrian,Crossing on pedestrian crossing facility,Crossing from driver's offside
9,66 - 75,Male,Pedestrian,Pedestrian,"In carriageway, crossing elsewhere",Crossing from driver's offside


DimCasualtyProfile
  Grain        : ['casualty_age_band', 'casualty_sex', 'casualty_class', 'casualty_type', 'pedestrian_location', 'pedestrian_movement']
  Distinct Rows: 2,477
  Unknown Rows : 0
  Expected Rows: 2,477
  Notes        : Grain includes pedestrian cols (FK to DimPedestrianContext)



### 5.3 DimVehicleEvent
**Grain**: `vehicle_type × driver_age_band × driver_sex × vehicle_manoeuvre × skidding_and_overturning × vehicle_leaving_carriageway`

In [29]:
grain_veh = [
    'vehicle_type', 'driver_age_band', 'driver_sex',
    'vehicle_manoeuvre', 'skidding_and_overturning', 'vehicle_leaving_carriageway'
]
veh_distinct = vehicles[grain_veh].drop_duplicates().reset_index(drop=True)

for col in grain_veh:
    vals = sorted(vehicles[col].dropna().unique())
    print(f'  [{col}] ({len(vals)} unique values): {vals}')
print()
display(veh_distinct.head(10))

profile(
    'DimVehicleEvent', vehicles, grain_veh,
    notes='6-col junk-style dimension for vehicle involvement'
)

  [vehicle_type] (19 unique values): ['Agricultural vehicle', 'Bus or coach (17 or more pass seats)', 'Car', 'Electric motorcycle', 'Goods 7.5 tonnes mgw and over', 'Goods over 3.5t. and under 7.5t', 'Minibus (8 - 16 passenger seats)', 'Mobility scooter', 'Motorcycle 125cc and under', 'Motorcycle 50cc and under', 'Motorcycle over 125cc and up to 500cc', 'Motorcycle over 500cc', 'Other vehicle', 'Pedal cycle', 'Ridden horse', 'Taxi/Private hire car', 'Tram', 'Unknown', 'Van / Goods 3.5 tonnes mgw or under']
  [driver_age_band] (12 unique values): ['0 - 5', '11 - 15', '16 - 20', '21 - 25', '26 - 35', '36 - 45', '46 - 55', '56 - 65', '6 - 10', '66 - 75', 'Over 75', 'Unknown']
  [driver_sex] (3 unique values): ['Female', 'Male', 'Unknown']
  [vehicle_manoeuvre] (18 unique values): ['Changing lane to left', 'Changing lane to right', 'Going ahead', 'Moving off', 'Over taking moving vehicle on its offside', 'Overtaking on nearside (passengers side nearest kerb)', 'Overtaking stationary vehicl

,vehicle_type,driver_age_band,driver_sex,vehicle_manoeuvre,skidding_and_overturning,vehicle_leaving_carriageway
0,Car,46 - 55,Male,Going ahead,NaN,Did not leave carriageway
1,Car,56 - 65,Female,Going ahead,NaN,Did not leave carriageway
2,Car,26 - 35,Male,Moving off,NaN,Did not leave carriageway
3,Car,26 - 35,Male,Going ahead,NaN,Did not leave carriageway
4,Car,26 - 35,Female,Reversing,NaN,Did not leave carriageway
5,Car,56 - 65,Male,Turning right,NaN,Did not leave carriageway
6,Car,36 - 45,Male,Going ahead,NaN,Did not leave carriageway
7,Car,26 - 35,Male,Going ahead,NaN,Straight ahead at junction
8,Motorcycle 125cc and under,46 - 55,Male,Going ahead,Overturned,Did not leave carriageway
9,Motorcycle 125cc and under,16 - 20,Male,Going ahead,Skidded,Offside


DimVehicleEvent
  Grain        : ['vehicle_type', 'driver_age_band', 'driver_sex', 'vehicle_manoeuvre', 'skidding_and_overturning', 'vehicle_leaving_carriageway']
  Distinct Rows: 20,578
  Unknown Rows : 9,451
  Expected Rows: 20,578
  Notes        : 6-col junk-style dimension for vehicle involvement



---
## 6. Chiều Mức độ

### Schema
| Bảng | Grain |
|---|---|
| DimSeverity | SeverityName – union của collision_severity và casualty_severity |

### 6.1 DimSeverity *(Role-playing Dimension)*
**Grain**: `SeverityName` – union từ `collision_severity` (Fact FK 1) và `casualty_severity` (Fact FK 2).

In [30]:
sev_col = collisions[['collision_severity']].rename(columns={'collision_severity': 'SeverityName'})
sev_cas = casualties[['casualty_severity']].rename(columns={'casualty_severity': 'SeverityName'})
sev_union = pd.concat([sev_col, sev_cas], ignore_index=True)
sev_distinct = sev_union.drop_duplicates().dropna().reset_index(drop=True)

print(f'collision_severity unique : {sorted(collisions["collision_severity"].dropna().unique())}')
print(f'casualty_severity unique  : {sorted(casualties["casualty_severity"].dropna().unique())}')

col_only = set(collisions['collision_severity'].dropna()) - set(casualties['casualty_severity'].dropna())
cas_only = set(casualties['casualty_severity'].dropna()) - set(collisions['collision_severity'].dropna())
overlap  = set(collisions['collision_severity'].dropna()) & set(casualties['casualty_severity'].dropna())
print(f'Collision-only  : {sorted(col_only)}')
print(f'Casualty-only   : {sorted(cas_only)}')
print(f'Overlap         : {sorted(overlap)}')
print()
display(sev_distinct)

profile(
    'DimSeverity', sev_union, ['SeverityName'],
    notes='Role-playing dim – union of collision_severity & casualty_severity'
)

collision_severity unique : ['Fatal', 'Serious', 'Slight']
casualty_severity unique  : ['Fatal', 'Serious', 'Slight']
Collision-only  : []
Casualty-only   : []
Overlap         : ['Fatal', 'Serious', 'Slight']



,SeverityName
0,Slight
1,Serious
2,Fatal


DimSeverity
  Grain        : ['SeverityName']
  Distinct Rows: 3
  Unknown Rows : 0
  Expected Rows: 3
  Notes        : Role-playing dim – union of collision_severity & casualty_severity



---
## 7. Báo cáo tổng hợp

In [31]:
summary_df = pd.DataFrame(results)[[
    'Dimension', 'Grain Columns', 'Distinct Rows', 'Unknown Rows', 'Expected Rows', 'Notes'
]]

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 20)

print('=' * 90)
print('DIMENSION PROFILING REPORT – STATS19 OLAP Schema (12 Dimensions)')
print('=' * 90)
display(summary_df)

DIMENSION PROFILING REPORT – STATS19 OLAP Schema (12 Dimensions)


,Dimension,Grain Columns,Distinct Rows,Unknown Rows,Expected Rows,Notes
0,DimDate,collision_date,1826,0,1826,1 row per unique date; key = YYYYMMDD INT; Key 0 for Unknown
1,DimTime,collision_time,1440,0,1440,1 row per unique HH:MM; max 1440 rows; FK -> DimTimeBand
2,DimTimeBand,TimeBandName (static),6,1,7,Static – 6 bands + 1 Unknown (Key=0) placeholder
3,DimLocalAuthority,local_authority_district_code,370,0,370,Code only – LocalAuthorityName & RegionName need ONS reference table (NOT in...
4,DimLocation,"longitude, latitude, local_authority_district_code",512359,33,512359,NULL coords allowed (DECIMAL NULL) – need Unknown placeholder row; FK -> Dim...
5,DimRoad,"first_road_class, road_type, speed_limit_mph, area_type",389,2,389,speed_limit_mph is numeric – NULL row counted as Unknown
6,DimEnvironment,"light_conditions, weather_conditions, road_surface_conditions",228,0,228,Junk dimension – cross product of 3 environment cols
7,DimJunction,"junction_detail, junction_control, pedestrian_crossing",295,0,295,Junk dimension – 3 junction / crossing cols
8,DimPedestrianContext,"pedestrian_location, pedestrian_movement",89,0,89,Sub-dimension of DimCasualtyProfile (Snowflake branch)
9,DimCasualtyProfile,"casualty_age_band, casualty_sex, casualty_class, casualty_type, pedestrian_l...",2477,0,2477,Grain includes pedestrian cols (FK to DimPedestrianContext)


### Format báo cáo theo yêu cầu

In [32]:
print('BANG PROFILING DIMENSION')
print('-' * 65)
print(f"{'Dimension':<25} {'Distinct Rows':>14} {'Unknown Rows':>13} {'Expected Rows':>14}")
print('-' * 65)
for _, row in summary_df.iterrows():
    print(f"{row['Dimension']:<25} {row['Distinct Rows']:>14,} "
          f"{row['Unknown Rows']:>13,} {row['Expected Rows']:>14,}")
print('-' * 65)

BANG PROFILING DIMENSION
-----------------------------------------------------------------
Dimension                  Distinct Rows  Unknown Rows  Expected Rows
-----------------------------------------------------------------
DimDate                            1,826             0          1,826
DimTime                            1,440             0          1,440
DimTimeBand                            6             1              7
DimLocalAuthority                    370             0            370
DimLocation                      512,359            33        512,359
DimRoad                              389             2            389
DimEnvironment                       228             0            228
DimJunction                          295             0            295
DimPedestrianContext                  89             0             89
DimCasualtyProfile                 2,477             0          2,477
DimVehicleEvent                   20,578         9,451         20,578
Dim

---
## 8. Export ra Excel

In [36]:
with pd.ExcelWriter(OUT_DIR / 'dimension_profiling_report.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Dimension Profiling', index=False)
    issues_df.to_excel(writer,  sheet_name='Schema Issues',       index=False)

print(f'Exported -> {OUT_DIR / "dimension_profiling_report.xlsx"}')

Exported -> D:\OLAP\Data\Profiling\dimension_profiling_report.xlsx
